# SMRD stage 1 — source-text encoder baseline (Colab)

Fine-tunes RoBERTa-base end-to-end on the source tweet only. Model selection uses
validation macro-F1; the test split is evaluated once per seed, on the full test set
and on the dedup-clean subset (test rows with no near-duplicate in train/val).

Checkpoints go to Drive after every epoch. If the runtime disconnects, re-run the
cells: each run resumes from its last finished epoch, and finished seeds are skipped.

Run order: Twitter16 → Twitter15 → PHEME.

In [ ]:
#@title 1. GPU check and dependencies
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('Runtime -> Change runtime type -> GPU (T4 or L4)')
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GiB')
%pip install -q 'transformers==5.22.1' 'tokenizers==0.23.3' 'safetensors==0.8.0'     'huggingface_hub==2.4.0' 'pandas==3.1.0' 'scikit-learn==1.9.1'

In [ ]:
#@title 2. Mount Drive and locate the project
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')

CKPT_ROOT = '/content/drive/MyDrive/smrd/checkpoints/encoder'
os.makedirs(CKPT_ROOT, exist_ok=True)

# Either upload the repository to Drive, or clone the stage1-encoder branch
# (the branch must be pushed to GitHub first).
PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/smrd-reproduction')
if not (PROJECT_ROOT / 'src' / 'run_encoder.py').exists():
    PROJECT_ROOT = pathlib.Path('/content/smrd-reproduction')
    if not PROJECT_ROOT.exists():
        !git clone -b stage1-encoder https://github.com/Miyonk/smrd-reproduction.git {PROJECT_ROOT}
assert (PROJECT_ROOT / 'src' / 'run_encoder.py').exists(), f'project not found at {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
print('project:', PROJECT_ROOT, '| checkpoints:', CKPT_ROOT)

In [ ]:
#@title 2b. Unit tests for the stage 1 code (CPU, tiny model, ~1 min)
%pip install -q 'pytest==9.2.0'
!python -m pytest tests/test_encoder_pipeline.py -q

In [ ]:
#@title 3. Build duplicate clusters + leakage report (writes data/splits/*_dup_clusters.json)
!python -m src.run_encoder dedup --all-datasets

In [ ]:
#@title 4. Smoke test: one epoch, one seed, Twitter16 (times one epoch for budgeting)
!python -m src.run_encoder train --dataset twitter16 --seeds 42 --max-epochs 1     --tag smoke --ckpt-root {CKPT_ROOT}

## Full runs

Defaults: batch 16, lr 2e-5, ≤ 6 epochs, early stopping on validation macro-F1 with
patience 3, label smoothing 0.1, class weights when the train split is imbalanced
(PHEME). Use the smoke-test epoch time to decide how many seeds fit in a session.

In [ ]:
#@title 5. Majority-class baselines
for ds in ('twitter16', 'twitter15', 'pheme'):
    !python -m src.run_encoder baseline --dataset {ds}

In [ ]:
#@title 6. Twitter16 (4-way), 3 seeds
!python -m src.run_encoder train --dataset twitter16 --seeds 42 43 44 --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 7. Twitter15 (4-way), 3 seeds
!python -m src.run_encoder train --dataset twitter15 --seeds 42 43 44 --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 8. PHEME (rumour vs non-rumour), 3 seeds
!python -m src.run_encoder train --dataset pheme --seeds 42 43 44 --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 9. Summaries (copied to Drive)
import json, glob, shutil
for p in sorted(glob.glob('results/stage1/*.json')):
    shutil.copy(p, CKPT_ROOT)
    s = json.load(open(p))
    if 'test_full' in s and 'accuracy' in s['test_full'] and isinstance(s['test_full']['accuracy'], dict):
        f, c = s['test_full'], s.get('test_clean', {})
        print(f"{s['run']:<36} full acc {f['accuracy']['mean']:.3f}±{f['accuracy']['std']:.3f} "
              f"macroF1 {f['macro_f1']['mean']:.3f} | clean acc {c['accuracy']['mean']:.3f} (n={c['n']})")